In [1]:
from matplotlib import pyplot as plt
import numpy as np

import torch
from sklearn.metrics import classification_report

from tqdm import tqdm

import time

from PieceDetection import PieceDetection

from Dataset.DataSetLoaders import ChessDataset

In [2]:
ds = ChessDataset.ChessDataset(
    config={
        "img_size": (640,640)
    }
)

In [3]:
pc_cnn = PieceDetection.PieceDetector("cnn")
# pc_cnn = PieceDetection.PieceDetector("cnn_onnx_static")
# pc_cnn = PieceDetection.PieceDetector("cnn_prunned")
pc_yolo = PieceDetection.PieceDetector("yolo")

In [ ]:
accs = []
avg_preprocess_time = 0
avg_process_time = 0
for img, label in tqdm(ds):
    try:
        pc_cnn.set_img(img, label["corners"])
        start_time = time.perf_counter()
        pc_cnn.preprocess()
        interm_time = time.perf_counter()
        preds = pc_cnn.predict()
        end_time = time.perf_counter()
        avg_preprocess_time += interm_time - start_time
        avg_process_time += end_time - interm_time

        acc = (preds.argmax(dim=1).squeeze() == label["board_tensor"]).sum().item()
        accs.append(acc)
    except:
        pass

avg_preprocess_time /= len(accs)
avg_process_time /= len(accs)
accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f} | Avg Pre-Processing Time: {avg_preprocess_time*1e3:.3f}ms | Avg Processing Time: {avg_process_time*1e3:.3f}ms")

  1%|          | 8/1009 [00:02<05:12,  3.20it/s]

In [4]:
accs = []
avg_preprocess_time = 0
avg_process_time = 0
for img, label in tqdm(ds):
    try:
        pc_yolo.set_img(img, label["corners"])
        start_time = time.perf_counter()
        pc_yolo.preprocess()
        interm_time = time.perf_counter()
        preds = pc_yolo.predict()
        end_time = time.perf_counter()
        avg_preprocess_time += interm_time - start_time
        avg_process_time += end_time - interm_time
        
        acc = (preds.squeeze() == label["board_tensor"]).sum().item()
        accs.append(acc)
    except:
        pass


avg_preprocess_time /= len(accs)
avg_process_time /= len(accs)
accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f} | Avg Pre-Processing Time: {avg_preprocess_time*1e3:.3f}ms | Avg Processing Time: {avg_process_time*1e3:.3f}ms")

100%|██████████| 1009/1009 [00:47<00:00, 21.09it/s]

Acc: 0.90 | Errors: 6.4975 | Avg Pre-Processing Time: 0.157ms | Avg Processing Time: 16.803ms


In [4]:
train_ds, valid_ds, test_ds = ChessDataset.ChessDataset.train_valid_test_split(ds, sizes=(.8,.1,.1), random_state=42)

In [5]:
accs = []
actual = []
preds_all = []
avg_preprocess_time = 0
avg_process_time = 0

for img, label in tqdm(test_ds):
    try:
        pc_cnn.set_img(img, label["corners"])
        start_time = time.perf_counter()
        pc_cnn.preprocess()
        interm_time = time.perf_counter()
        preds = pc_cnn.predict()
        end_time = time.perf_counter()
        avg_preprocess_time += interm_time - start_time
        avg_process_time += end_time - interm_time

        acc = (preds.argmax(dim=1).squeeze() == label["board_tensor"]).sum().item()
        preds_all.append(preds.argmax(dim=1).reshape(-1))
        actual.append(label["board_tensor"].reshape(-1))
        accs.append(acc)
    except:
        pass

avg_preprocess_time /= len(accs)
avg_process_time /= len(accs)
accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f} | Avg Pre-Processing Time: {avg_preprocess_time*1e3:.3f}ms | Avg Processing Time: {avg_process_time*1e3:.3f}ms")

actual = torch.cat(actual, dim=0)
preds_all = torch.cat(preds_all, dim=0)
rep = classification_report(actual, preds_all, target_names=['White-Pawn', 'White-Knight', 'White-Bishop', 'White-Rook', 'White-Queen', 'White-King', 'Black-Pawn', 'Black-Knight', 'Black-Bishop', 'Black-Rook', 'Black-Queen', 'Black-King', 'Empty'])
print(rep)

100%|██████████| 103/103 [00:43<00:00,  2.39it/s]

Acc: 0.99 | Errors: 0.6990 | Avg Pre-Processing Time: 22.191ms | Avg Processing Time: 375.747ms
              precision    recall  f1-score   support

  White-Pawn       0.99      0.98      0.98       586
White-Knight       0.96      0.97      0.97       106
White-Bishop       0.95      0.98      0.97       106
  White-Rook       0.99      0.98      0.98       161
 White-Queen       0.93      1.00      0.96        76
  White-King       1.00      0.95      0.98       103
  Black-Pawn       0.99      0.98      0.98       602
Black-Knight       0.93      0.99      0.96       106
Black-Bishop       0.95      0.93      0.94        96
  Black-Rook       0.99      0.94      0.97       163
 Black-Queen       0.97      0.99      0.98        74
  Black-King       0.95      0.97      0.96       103
       Empty       1.00      1.00      1.00      4310

    accuracy                           0.99      6592
   macro avg       0.97      0.97      0.97      6592
weighted avg       0.99      0.99     

In [6]:
accs = []
actual = []
preds_all = []
for img, label in tqdm(test_ds):
    try:
        pc_yolo.set_img(img, label["corners"])
        pc_yolo.preprocess()
        preds = pc_yolo.predict()

        acc = (preds.squeeze() == label["board_tensor"]).sum().item()
        preds_all.append(preds.reshape(-1))
        actual.append(label["board_tensor"].reshape(-1))
        accs.append(acc)
    except:
        pass

accs = np.array(accs)
print(f"Acc: {accs.mean() / 64:.2f} | Errors: {(64 - accs).mean():.4f}")

actual = torch.cat(actual, dim=0)
preds_all = torch.cat(preds_all, dim=0)
rep = classification_report(actual, preds_all)
print(rep)

  0%|          | 0/103 [00:00<?, ?it/s]/mnt/D/University/Thesis/.venv/lib/python3.13/site-packages/torch/cuda/__init__.py:174: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109.)
  return torch._C._cuda_getDeviceCount() > 0
100%|██████████| 103/103 [00:14<00:00,  7.08it/s]

Acc: 0.91 | Errors: 5.5825
              precision    recall  f1-score   support

           0       0.98      0.68      0.80       586
           1       0.35      0.82      0.49       106
           2       0.78      0.62      0.69       106
           3       0.93      0.71      0.81       161
           4       0.90      0.62      0.73        76
           5       0.93      0.80      0.86       103
           6       0.97      0.75      0.84       602
           7       0.51      0.98      0.68       106
           8       0.63      0.83      0.72        96
           9       0.76      0.89      0.82       163
          10       0.90      0.74      0.81        74
          11       0.78      0.90      0.83       103
          12       0.97      1.00      0.98      4310

    accuracy                           0.91      6592
   macro avg       0.80      0.80      0.78      6592
weighted avg       0.93      0.91      0.92      6592

